In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

# 1. Germanų kalbos

germanic_languages = [
    'eng',  # English
    'deu',  # German
    'nld',  # Dutch
    'swe',  # Swedish
    'afr',  # Afrikaans
    'dan',  # Danish
    'nor',  # Norwegian
    'nds',  # Low German
    'yid',  # Yiddish
    'sco',  # Scots
    'fry',  # Frisian
    'ltz',  # Luxembourgish
    'isl',  # Icelandic
    'fao',  # Faroese
]


# Zero-shot labels 
candidate_labels = [
    "English", "German", "Dutch", "Swedish", "Afrikaans",
    "Danish", "Norwegian", "Low German", "Yiddish", "Scots",
    "Frisian", "Luxembourgish", "Icelandic", "Faroese"
]

# Mapping English → ISO 
label_to_iso = {
    "English": "eng",
    "German": "deu",
    "Dutch": "nld",
    "Swedish": "swe",
    "Afrikaans": "afr",
    "Danish": "dan",
    "Norwegian": "nor",
    "Low German": "nds",
    "Yiddish": "yid",
    "Scots": "sco",
    "Frisian": "fry",
    "Luxembourgish": "ltz",
    "Icelandic": "isl",
    "Faroese": "fao"
}


# 2. Dataset

df = pd.read_csv("hf://datasets/agentlans/tatoeba-english-translations/All.csv.gz")
df = df[df["Language"].isin(germanic_languages)]

print("Pradiniai dydžiai:")
print(df["Language"].value_counts())


# Subalansuoti

min_count = df["Language"].value_counts().min()
df_bal = pd.concat(
    [g.sample(min_count, random_state=123) for _, g in df.groupby("Language")],
    ignore_index=True,
)

df_bal = df_bal.drop_duplicates(subset=["Translation"])

print("\nSubalansuotos klasės:")
print(df_bal["Language"].value_counts())


# 3. Split


X = df_bal["Translation"].tolist()
y = df_bal["Language"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)


# 4. Zero-shot transformeris


print("\nKraunamas zero-shot modelis...")
pipe = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)


# 5. PROGNOZĖS


y_pred = []

for text in X_test:

    out = pipe(text, candidate_labels)

    best_label = out["labels"][0]           
    iso_code   = label_to_iso[best_label]   

    y_pred.append(iso_code)


# 6. Rezultatai

acc = accuracy_score(y_test, y_pred)

print("\n======================================")
print("### ZERO-SHOT TRANSFORMERIO REZULTATAI (GERMANIC) ###")
print("======================================\n")

print("Tikslumas:", acc)
print("\nKlasifikavimo ataskaita:")
print(classification_report(y_test, y_pred, zero_division=0))


ERROR! Session/line number was not unique in database. History logging moved to new session 29
Pradiniai dydžiai:
Language
deu    530145
nld    156354
dan     47250
swe     40465
yid     25477
isl     11321
nds      6478
afr      2986
ltz      1410
fao       390
fry       369
sco        94
Name: count, dtype: int64

Subalansuotos klasės:
Language
dan    94
deu    94
fao    94
isl    94
yid    94
nds    94
swe    94
nld    94
afr    93
ltz    92
fry    89
sco    85
Name: count, dtype: int64

Kraunamas zero-shot modelis...


config.json: 0.00B [00:00, ?B/s]

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled fo

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu



### ZERO-SHOT TRANSFORMERIO REZULTATAI (GERMANIC) ###

Tikslumas: 0.336322869955157

Klasifikavimo ataskaita:
              precision    recall  f1-score   support

         afr       0.00      0.00      0.00        16
         dan       0.50      0.12      0.20        24
         deu       0.36      0.73      0.48        22
         eng       0.00      0.00      0.00         0
         fao       0.00      0.00      0.00        21
         fry       0.00      0.00      0.00        14
         isl       0.45      0.88      0.60        17
         ltz       0.00      0.00      0.00        19
         nds       0.08      0.06      0.07        18
         nld       0.31      0.36      0.33        14
         nor       0.00      0.00      0.00         0
         sco       0.15      0.29      0.20        17
         swe       0.79      0.50      0.61        22
         yid       0.54      1.00      0.70        19

    accuracy                           0.34       223
   macro avg       0.23

In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

df = pd.read_csv("hf://datasets/agentlans/tatoeba-english-translations/All.csv.gz")

languages_list = ['deu', 'nld', 'swe', 'fry']

filtered_df = df[df["Language"].isin(languages_list)]

print("Pradinės klasės:")
print(filtered_df["Language"].value_counts())


# 2. Subalansuoti pagal mažiausia


min_count = filtered_df["Language"].value_counts().min()
print("\nMažiausias klasės dydis:", min_count)

df_balanced = pd.concat(
    [
        group.sample(min_count, random_state=123)
        for _, group in filtered_df.groupby("Language")
    ],
    ignore_index=True
)

print("\nSubalansuotos klasės:")
print(df_balanced["Language"].value_counts())

X = df_balanced["Translation"].tolist()
y = df_balanced["Language"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)


# Transformeris


model_name = "Mike0307/multilingual-e5-language-detection"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)


# Mapping LABEL_X → ISO 639-3
transformer_map = {
    17: "deu",
    10: "nld",
    15: "fry",
    39: "swe"
}

# Prognozes


y_pred = []

for text in X_test:
    out = pipe(text, top_k=1)[0]
    label_num = int(out["label"].replace("LABEL_", ""))

    pred_lang = transformer_map.get(label_num, "UNKNOWN")
    y_pred.append(pred_lang)


# Rezultatai

acc = accuracy_score(y_test, y_pred)

print("\n======================================")
print("### TRANSFORMERIO REZULTATAI ###")
print("======================================\n")

print("Tikslumas:", acc)
print("\nKlasifikavimo ataskaita:")
print(classification_report(y_test, y_pred))


Pradinės klasės:
Language
deu    530145
nld    156354
swe     40465
fry       369
Name: count, dtype: int64

Mažiausias klasės dydis: 369

Subalansuotos klasės:
Language
deu    369
fry    369
nld    369
swe    369
Name: count, dtype: int64


Device set to use cpu



### TRANSFORMERIO REZULTATAI ###

Tikslumas: 0.9391891891891891

Klasifikavimo ataskaita:
              precision    recall  f1-score   support

     UNKNOWN       0.00      0.00      0.00         0
         deu       0.99      1.00      0.99        78
         fry       1.00      0.80      0.89        65
         nld       0.94      0.98      0.96        83
         swe       1.00      0.96      0.98        70

    accuracy                           0.94       296
   macro avg       0.79      0.75      0.76       296
weighted avg       0.98      0.94      0.96       296



C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: